In [39]:
# imports
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm
import matplotlib.pyplot as plt
from sklearn.linear_model import Lasso, LogisticRegression, LassoCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, 
    mean_squared_error, 
    confusion_matrix, 
    ConfusionMatrixDisplay, 
    classification_report, 
    roc_curve, 
    roc_auc_score
)
from sklearn.utils import resample
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [40]:
# function that drops derived columns
def drop_derived(df):
    drop_list = ['derived_ethnicity', 'derived_race', 'derived_sex']
    columns_to_drop = [col for col in df.columns if any(prefix in col for prefix in drop_list)]
    df = df.drop(columns=columns_to_drop)
    return df 

In [41]:
# function that splits test and training
def split_test_training(df):
    
    # split the data
    X_train, X_test, y_train, y_test = train_test_split(df.drop('action_taken', axis=1), df['action_taken'], test_size=0.2, random_state=42)
    
    # scale data
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # add a constant
    X_train_scaled = sm.add_constant(X_train_scaled, has_constant='add')
    X_test_scaled = sm.add_constant(X_test_scaled, has_constant='add')
    
    return X_train_scaled, X_test_scaled, y_train, y_test

In [42]:
# function that regularizes the logisitc regression
def logistic_regression(X_train, y_train):
    
    # set up model
    pipeline = Pipeline([
        ('logistic', LogisticRegression(penalty='l1', solver='saga', max_iter=10000))])
    parameters = {'logistic__C': np.logspace(-3, 3, 10)}
    grid_search = GridSearchCV(pipeline, parameters, cv=3, scoring='accuracy', n_jobs=-1)
    grid_search.fit(X_train, y_train)
    
    # print best accuracy and c_val
    print('Best trained cross-validated accuracy:', grid_search.best_score_)
    print('Best c_value:', grid_search.best_params_['logistic__C'])
    
    # best model
    best_pipeline = grid_search.best_estimator_
    return best_pipeline, grid_search.best_params_['logistic__C']

In [43]:
# function that makes the classification report a png
def classificationreport(best_model, X_test, y_test, year):
    
    # find the prediction and class report
    y_pred = best_model.predict(X_test)
    class_report = classification_report(y_test, y_pred)
    
    # plot the classification report
    plt.figure(figsize=(10, 10))
    plt.text(0.01, 1.25, 'Classification Report ' + year, {'fontsize': 16}, fontproperties='monospace')
    plt.text(0.01, 0.05, class_report, {'fontsize': 10}, fontproperties='monospace')
    plt.axis('off')
    
    # make it into a png
    plt.savefig(f'classification_report_{year}_noderived.png')
    plt.close()

In [44]:
# function that organizes the coefficients into a df then csv
def make_coefficient_df(best_model, df, year):
    feature_names = df.drop('action_taken', axis=1).columns.tolist()
    feature_names = ['const'] + feature_names
    coefficients = best_model.named_steps['logistic'].coef_[0]
    coefficients_df = pd.DataFrame({'Feature': feature_names, 'Coefficient': coefficients})
    coefficients_df.to_csv(f'coefficient_df_{year}_noderived.csv', index=False)

In [45]:
# function that makes the confusion matrix a png
def print_cm_model(best_model, X_test, y_test, year):
    
    # find the prediction and confusion matrix
    y_pred = best_model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap=plt.cm.Blues, normalize='all')
    disp.ax_.set_title(f'Confusion Matrix for {year} Model')
    
    # make it into a png
    plt.savefig(f'confusion_matrix_{year}_noderived.png')
    plt.close()

In [46]:
# function that makes the roc curve a png
def plot_roc_curve(best_model, X_test, y_test, year):
    
    # find the prediction, roc_curve, auc score
    y_prob = best_model.predict_proba(X_test)[:, 1]
    fpr, tpr, thresholds = roc_curve(y_test, y_prob)
    auc_score = roc_auc_score(y_test, y_prob)

    # plot the roc curve
    plt.figure()
    plt.plot(fpr, tpr, color='blue', label='ROC curve (area = %0.2f)' % auc_score)
    plt.plot([0, 1], [0, 1], color='grey', linestyle='--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'Receiver Operating Characteristic (ROC) Curve {year}')
    plt.legend(loc='lower right')

    # make it into a png
    plt.savefig(f'roc_curve_{year}_noderived.png')
    plt.close()

In [47]:
#2018
dummy_2018 = pd.read_csv('dummy_2018.csv')
dropped_2018 = drop_derived(dummy_2018)
X_train_2018, X_test_2018, y_train_2018, y_test_2018 = split_test_training(dropped_2018)
model_2018, best_c_2018 = logistic_regression(X_train_2018, y_train_2018)
classificationreport(model_2018, X_test_2018, y_test_2018, '2018')
make_coefficient_df(model_2018, dropped_2018, '2018')
print_cm_model(model_2018, X_test_2018, y_test_2018, '2018')
plot_roc_curve(model_2018, X_test_2018, y_test_2018, '2018')

Best trained cross-validated accuracy: 0.9972846133320776
Best c_value: 0.1


In [48]:
#2019
dummy_2019 = pd.read_csv('dummy_2019.csv')
dropped_2019 = drop_derived(dummy_2019)
X_train_2019, X_test_2019, y_train_2019, y_test_2019 = split_test_training(dropped_2019)
model_2019, best_c_2019 = logistic_regression(X_train_2019, y_train_2019)
classificationreport(model_2019, X_test_2019, y_test_2019, '2019')
make_coefficient_df(model_2019, dropped_2019, '2019')
print_cm_model(model_2019, X_test_2019, y_test_2019, '2019')
plot_roc_curve(model_2019, X_test_2019, y_test_2019, '2019')

Best trained cross-validated accuracy: 0.9964731962918177
Best c_value: 0.46415888336127775


In [49]:
#2020
dummy_2020 = pd.read_csv('dummy_2020.csv')
dropped_2020 = drop_derived(dummy_2020)
X_train_2020, X_test_2020, y_train_2020, y_test_2020 = split_test_training(dropped_2020)
model_2020, best_c_2020 = logistic_regression(X_train_2020, y_train_2020)
classificationreport(model_2020, X_test_2020, y_test_2020, '2020')
make_coefficient_df(model_2020, dropped_2020, '2020')
print_cm_model(model_2020, X_test_2020, y_test_2020, '2020')
plot_roc_curve(model_2020, X_test_2020, y_test_2020, '2020')

Best trained cross-validated accuracy: 0.9976113698793743
Best c_value: 0.46415888336127775


In [50]:
#2021
dummy_2021 = pd.read_csv('dummy_2021.csv')
dropped_2021 = drop_derived(dummy_2021)
X_train_2021, X_test_2021, y_train_2021, y_test_2021 = split_test_training(dropped_2021)
model_2021, best_c_2021 = logistic_regression(X_train_2021, y_train_2021)
classificationreport(model_2021, X_test_2021, y_test_2021, '2021')
make_coefficient_df(model_2021, dropped_2021, '2021')
print_cm_model(model_2021, X_test_2021, y_test_2021, '2021')
plot_roc_curve(model_2021, X_test_2021, y_test_2021, '2021')

Best trained cross-validated accuracy: 0.999163010435144
Best c_value: 0.46415888336127775


In [51]:
#2022
dummy_2022 = pd.read_csv('dummy_2022.csv')
dropped_2022 = drop_derived(dummy_2022)
X_train_2022, X_test_2022, y_train_2022, y_test_2022 = split_test_training(dropped_2022)
model_2022, best_c_2022 = logistic_regression(X_train_2022, y_train_2022)
classificationreport(model_2022, X_test_2022, y_test_2022, '2022')
make_coefficient_df(model_2022, dropped_2022, '2022')
print_cm_model(model_2022, X_test_2022, y_test_2022, '2022')
plot_roc_curve(model_2022, X_test_2022, y_test_2022, '2022')

Best trained cross-validated accuracy: 0.9992946547655689
Best c_value: 10.0
